# M²S²L Architecture Overview

M²S²L is an **unsupervised reconstruction-based Video Anomaly Detection (VAD)** framework built upon **Mamba**. The main motivation is that anomalies can occur at **different spatial scales**, **different temporal durations**, and affect **appearance** and **motion** differently. Previous Mamba-based approaches generally learn a single representation, limiting their ability to model these diverse anomaly patterns.

To address this limitation, M²S²L introduces three key components:

## 1. Multi-scale Spatial Learning

Instead of extracting features from a single image resolution, the model processes the input using **three spatial scales** (fine, medium, and coarse).

Each scale captures complementary information:

- **Fine scale:** local textures and small anomalies.
- **Medium scale:** balances local details and global context.
- **Coarse scale:** captures large objects and scene-level structures.

The three spatial representations are then fused through adaptive weighting, allowing the network to automatically determine which scale is most informative for each input.

---

## 2. Multi-scale Temporal Learning

Temporal anomalies may occur over different durations. Therefore, M²S²L models motion at **three temporal scales**:

- Short-term motion
- Medium-term motion
- Long-term behavior

Instead of using computationally expensive optical flow, the model computes **frame differences** as motion representations, which are processed by Temporal Mamba Blocks. The temporal features are finally fused using an attention mechanism that learns the relative importance of each temporal scale.

---

## 3. Feature Decomposition

After combining the spatial and temporal features, M²S²L decomposes the fused representation into:

- Common features ($F_{common}$)
- Appearance-specific features ($F_{app}$)
- Motion-specific features ($F_{motion}$)

This separation allows each decoder to specialize in reconstructing its corresponding modality rather than forcing a single representation to model both tasks simultaneously.

To further improve anomaly detection, the model introduces **three memory banks** that store prototypes of normal:

- Common features ($M_{c}$)
- Appearance features ($M_{a}$)
- Motion features ($M_{m}$)

During inference, anomalous samples cannot be well reconstructed from these normal prototypes, producing larger reconstruction errors that are used to identify anomalies.

---

## Anomaly Scoring

The final anomaly score combines the reconstruction quality of both appearance and motion.

The procedure is:

1. Compute frame reconstruction error.
2. Compute motion reconstruction error.
3. Convert both errors into **PSNR** values.
4. Compute a weighted combination:

$$
PSNR_{combined}
=
\alpha PSNR_{frame}
+
(1-\alpha)PSNR_{motion}
$$

where **α** balances the contribution of appearance and motion.

Finally, anomaly scores are obtained through min-max normalization across the entire test sequence.

---

## Results

| Method    |     Ped2 |   Avenue | ShanghaiTech |    FLOPs |   Params |    FPS |
| --------- | -------: | -------: | -----------: | -------: | -------: | -----: |
| VADMamba  | **98.5** |     91.5 |         77.0 |        — |    28.1M | **90** |
| STNMamba  |     98.0 |     89.0 |         74.9 | **1.5G** | **7.2M** |     40 |
| **M²S²L** | **98.5** | **92.1** |     **77.9** |    20.1G |    14.9M |     45 |

**Main observations**

- **Highest AUC on Avenue (92.1%)** and **ShanghaiTech (77.9%)** among the compared Mamba-based methods.
- Matches **VADMamba** on **UCSD Ped2 (98.5%)** while outperforming it on the more challenging datasets.
- Compared with **STNMamba**, the gains are:
    - **+0.5%** on Ped2
    - **+3.1%** on Avenue
    - **+3.0%** on ShanghaiTech
- Although it is not the fastest model, **45** FPS is still sufficient for real-time inference while achieving better detection accuracy.

---

## Key Idea

Unlike previous Mamba-based VAD methods, M²S²L explicitly specializes feature learning along **two dimensions**:

- **Scale specialization:** captures anomalies ranging from small local changes to large scene-level events.
- **Modality specialization:** independently models appearance and motion while still leveraging their shared information.

This design enables richer spatial-temporal representations while preserving the computational efficiency of Mamba, resulting in improved detection performance, particularly on challenging datasets such as **CUHK Avenue** and **ShanghaiTech**.

---

## Comparison with previous models

| Modelo       | Spatial branch    | Temporal branch    | Feature decomposition | Multi-scale             |
| ------------ | ----------------- | ------------------ | --------------------- | ----------------------- |
| **VADMamba** | ❌ (una sola rama) | ✅ (Mamba temporal) | ❌                     | ❌                       |
| **STNMamba** | ✅                 | ✅                  | ❌                     | ❌                       |
| **M²S²L**    | ✅                 | ✅                  | ✅                     | ✅ (espacial + temporal) |
